# День 6 — Инференс и сравнение моделей

Цель: сравнить baseline из Дня 4 и fine-tuned модель из Дня 5.

## Задача 1: загрузка моделей

Fine-tuned — из `models/fine_tuned_model`, baseline (LogisticRegression на
CLS-эмбеддингах) — из `models/baseline_model.pkl`. Для эмбеддингов baseline
нужна обычная предобученная модель `AutoModel`.

In [ ]:
from pathlib import Path

from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

from sentiment_lab.baseline import load_model as load_baseline_model

model_dir = Path("models/fine_tuned_model")
model_ft = AutoModelForSequenceClassification.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model_ft.eval()

baseline_model = load_baseline_model()
encoder_model = AutoModel.from_pretrained("distilbert-base-uncased")
encoder_model.eval()
print(f'{type(model_ft).__name__} / {type(baseline_model).__name__}')

## Задачи 2–3: функции предсказания

`predict_fine_tuned` и `predict_baseline` лежат в `sentiment_lab.inference`.
Версия baseline из задания ожидает TF-IDF `vectorizer`; здесь baseline построен
на CLS-эмбеддингах, поэтому функция получает `tokenizer` и `encoder_model`.

In [ ]:
from sentiment_lab.inference import predict_baseline, predict_fine_tuned

## Задача 4: сравнение на примерах

In [ ]:
test_texts = [
    "This movie was absolutely fantastic!",
    "Terrible, waste of my time.",
    "It was okay, nothing special.",
    "Best film I've seen this year!",
    "Boring and too long."
]

preds_ft = predict_fine_tuned(test_texts, model_ft, tokenizer)
preds_baseline = predict_baseline(test_texts, baseline_model, tokenizer, encoder_model)

for ft, base in zip(preds_ft, preds_baseline):
    print(f'\nТекст: {ft.text}')
    print(f'Fine-tuned: {ft.prediction} (probs: {ft.probabilities})')
    print(f'Baseline: {base.prediction} (probs: {base.probabilities})')
    print(f'Совпадают: {ft.prediction == base.prediction}')

## Задача 5: confusion matrix для обеих моделей

Тестовая выборка — `data/imdb_test.csv` (500 строк для скорости на CPU).
Матрицы сохраняются в `outputs/`.

In [ ]:
import pandas as pd

test_df = pd.read_csv("data/imdb_test.csv").sample(n=500, random_state=42)
test_texts_all = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

preds_ft_all = predict_fine_tuned(test_texts_all, model_ft, tokenizer)
y_pred_ft = [p.prediction for p in preds_ft_all]
preds_base_all = predict_baseline(test_texts_all, baseline_model, tokenizer, encoder_model)
y_pred_base = [p.prediction for p in preds_base_all]
print(f'Предсказаний: {len(y_pred_ft)} / {len(y_pred_base)}')

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

from sentiment_lab.evaluation import plot_confusion_matrix

cm_ft = plot_confusion_matrix(test_labels, y_pred_ft, labels=[0, 1], name="fine_tuned")
cm_base = plot_confusion_matrix(test_labels, y_pred_base, labels=[0, 1], name="baseline")

for name, y_pred in [("fine-tuned", y_pred_ft), ("baseline", y_pred_base)]:
    acc = accuracy_score(test_labels, y_pred)
    f1 = f1_score(test_labels, y_pred, average="macro")
    print(f'{name}: accuracy={acc:.3f}, macro F1={f1:.3f}')
print(cm_ft, cm_base, sep='\n')